<a href="https://colab.research.google.com/github/siddumais/starter-notebook/blob/main/work/notebooks/w01_research_question.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-02 — Research Question and Provisional Lane

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [4]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("../..")  # w01 lives in work/notebooks/, so go up two levels to repo root

import pandas as pd
print("Working directory:", os.getcwd())
print("CSV present:", os.path.exists("data/raw/content_refresh_anonymized.csv"))

Working directory: /content/flyrank-ml-internship-starter
CSV present: True


## 1. My lane (or freestyle) and why

*Name your lane — or say 'freestyle' and describe your own question. One short paragraph: why this one?*

**Provisional lane: Lane 2 — Refresh / Content Opportunity Scoring.**

I'm picking this because I already have two weeks of evidence pointing here, not because it's the default. In Week 1's discovery notebook I found that `content_type` moves CTR even at the *same* position tier (comparison articles underperform keyword articles at matched positions), which means position alone can't explain what's worth reviewing. In Week 2 I built a one-line hand rule (`stale x visible`) and a depth-2 decision tree against `is_declining_label`, and the two methods split the win — hand rule ahead at Precision@20, tree ahead at Precision@50 — which is exactly the shape of problem this lane is built for: a fixed rule and a learned ranking trading places depending on how deep the reviewer needs to go. That's a real signal worth 7 more weeks, not a coin flip. I'll confirm or drop this by end of Week 4, per the guide.

In [5]:
lane = "Refresh / Content Opportunity Scoring"
print("Provisional lane:", lane)

Provisional lane: Refresh / Content Opportunity Scoring


## 2. The question: decision, action, cost of a wrong call

*What decision does your work improve? Who acts on it? What does a wrong recommendation cost?*

**Research question:** Given limited weekly review capacity, which existing pages should a content editor look at first — for refresh, expansion, protection, pruning, or monitoring?

**Decision improved:** Right now an editor either reviews pages in an arbitrary order (newest first, alphabetical, whatever's on top of a spreadsheet) or uses a single hand-tuned rule. My work replaces that with a ranked queue backed by reason codes, so the editor's first 20-50 reviews are the highest-evidence candidates, not a guess.

**Who acts, and how:** A content/SEO editor with a fixed number of review slots per week. They open the top of the queue, read the reason codes (e.g. `declining_with_demand`, `low_ctr_visible_page`), and decide: refresh the content, fix metadata/CTR, leave it alone, or flag it for deeper review.

**Cost of a wrong call:**
- *False positive* (flagged high, not actually a problem): wastes an editor's limited hour on a page that didn't need attention — the scarce resource here is editor time, not compute.
- *False negative* (missed, but actually declining with real traffic at stake): a page that's genuinely losing visibility keeps sliding for another review cycle before anyone notices, which is the more expensive miss since it compounds — lost impressions this week are lost impressions next week too.
- Because both error types cost real time or real traffic, Precision@K (does the top of the list actually deserve to be there) matters more here than a global accuracy number an editor will never see end-to-end.

**Why data/ML helps at all:** A single hand rule (`stale x visible`) already showed real signal in Week 2 — but it also showed its ceiling: it won at Precision@20 and lost at Precision@50, meaning it runs out of nuance past the very top of the list. The signals that matter (staleness, visibility, position, CTR, engagement) interact — e.g. Week 1 showed CTR depends on `content_type` even after controlling for position — in ways too tangled to hand-write past a handful of if/else branches. That's the gap a model earns its place filling.

## 3. Quick look at the data (2-3 real numbers)

*Load the starter CSV below and show 2-3 real numbers that make your lane look worth the next 7 weeks.*

In [7]:
import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
n = len(df)

declining_with_demand = ((df["trend_direction"].str.lower() == "down") & (df["impressions_90d"] >= 100)).sum()
stale_visible_page   = ((df["days_since_last_update"] >= 180) & (df["impressions_90d"] >= 500)).sum()
low_ctr_visible_page = ((df["impressions_90d"] >= 500) & (df["avg_position"] > 0) & (df["avg_position"] <= 20) & (df["ctr"] < 0.5)).sum()

print(f"total pages in starter slice: {n:,}")
print(f"declining_with_demand (trend=down AND impressions_90d>=100): {declining_with_demand:,}  ({declining_with_demand/n:.1%})")
print(f"stale_visible_page (days_since_update>=180 AND impressions_90d>=500): {stale_visible_page:,}  ({stale_visible_page/n:.1%})")
print(f"low_ctr_visible_page (visible, top-20 position, ctr<0.5): {low_ctr_visible_page:,}  ({low_ctr_visible_page/n:.1%})")
print()
print("days_since_last_update, median:", df["days_since_last_update"].median(), "| max:", df["days_since_last_update"].max())

total pages in starter slice: 30,000
declining_with_demand (trend=down AND impressions_90d>=100): 13,152  (43.8%)
stale_visible_page (days_since_update>=180 AND impressions_90d>=500): 17  (0.1%)
low_ctr_visible_page (visible, top-20 position, ctr<0.5): 9,759  (32.5%)

days_since_last_update, median: 20.0 | max: 373


**What these numbers say:**

- `declining_with_demand` covers **43.8%** of the slice (13,152 of 30,000 pages) — a large, real candidate pool, not a fringe case.
- `stale_visible_page` (the exact staleness bar from the guide) covers only **0.1%** (17 pages) — median `days_since_last_update` in this slice is 20 days, so almost nothing here is "stale" by a 180-day bar. That's a genuine finding, not a bug: staleness alone won't be my primary signal — I'll need to define my target more around decline + demand + CTR than around raw staleness, or loosen/rethink the staleness threshold before I lean on it.
- `low_ctr_visible_page` covers **32.5%** (9,759 pages) — a second sizeable, independent opportunity pool, consistent with Week 1's finding that CTR varies by `content_type` at the same position.
- From the repo's own committed pipeline run (`outputs/model_report.md`, client-holdout validated): the baseline rule scores **0.240** Precision@50 (~12 of the top 50 are real) vs. the random forest's **0.740** (~37 of the top 50) — the same shape of gap I saw by hand in Week 2, now confirmed at a bigger, honestly-validated scale. That gap is the evidence this lane is worth building on.

## 4. Careful words: what I can and can't claim

*Write what your work will be able to say (observed, directional, decision-support) — and what it never will (causal proof, 'predicting Google').*

**What I can claim:**
- Observed, directional patterns: e.g. "pages with X combination of signals were more likely to be labeled declining in this window."
- Decision-support: a ranked review queue with reason codes an editor can inspect and override.
- Validated lift over a transparent baseline, measured with Precision@K and a client-holdout split (never in-sample numbers as the headline).

**What I will never claim:**
- That a refresh *caused* a recovery — that needs an experiment or causal design, which this data doesn't give me.
- That I've reverse-engineered any Google ranking factor, or AI citation/ranking behavior.
- That the current proxy label (`trend_direction == "down"`, a bucket computed from the current window) is the same as a true future outcome — a stronger version of this label, built from a prior-90-days -> next-30-days window, is the target I'm working toward, not the beginner label I'm stuck with.
- That any single score (like a combined `final_refresh_score`) replaces a human reviewer's judgment — the reason codes exist precisely so a person checks the *why*, not just the number.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.